# Semantic Search & LLM Application
**Starter Notebook**

Notebook ini adalah kerangka awal untuk membangun prototype aplikasi semantic search yang terintegrasi dengan LLM, menggunakan knowledge base FAQ e-commerce.

Alur besar: Load Knowledge Base → Embedding → Vector Database (FAISS) → Semantic Search → Prompt Design → LLM Integration → Uji Relevansi.

Isi tiap section sesuai instruksi di Assignment Brief. Jangan ubah struktur section, tapi silakan tambah cell baru di dalam tiap section jika diperlukan.

## 1. Import Library & Load Knowledge Base

In [3]:
import pandas as pd
import numpy as np
import faiss
import os
from dotenv import load_dotenv
from google import genai
from google.genai import types

load_dotenv('../.env')
client = genai.Client(api_key=os.environ.get('GEMINI_API_KEY'))

df = pd.read_csv('../data/ecommerce_faq_knowledge_base.csv')
print('Total entri knowledge base:', len(df))
df.head()

Total entri knowledge base: 59


,faq_id,category,question,answer
0,1,Pemesanan,Bagaimana cara melakukan pemesanan di platform...,"Pilih produk yang diinginkan, klik 'Tambah ke ..."
1,2,Pemesanan,Apakah saya bisa memesan tanpa membuat akun?,Saat ini pemesanan hanya bisa dilakukan setela...
2,3,Pemesanan,Bagaimana cara membatalkan pesanan yang sudah ...,Pesanan dapat dibatalkan melalui halaman 'Pesa...
3,4,Pemesanan,Berapa lama waktu yang dibutuhkan sebelum pesa...,Penjual biasanya memproses pesanan dalam waktu...
4,5,Pemesanan,Apakah saya bisa mengubah alamat pengiriman se...,Perubahan alamat hanya bisa dilakukan sebelum ...


## 2. Menyiapkan Unit Retrievable

Setiap entri FAQ (question + answer) diperlakukan sebagai satu unit yang akan di-embed dan disimpan ke vector database. Gabungkan question dan answer menjadi satu teks representatif per entri.

In [4]:
df['content'] = 'Q: ' + df['question'] + ' A: ' + df['answer']
documents = df['content'].tolist()
print('Contoh unit retrievable:')
print(documents[0])

Contoh unit retrievable:
Q: Bagaimana cara melakukan pemesanan di platform ini? A: Pilih produk yang diinginkan, klik 'Tambah ke Keranjang', lalu lanjutkan ke halaman checkout untuk mengisi alamat pengiriman dan metode pembayaran.


## 3. Membuat Embeddings

Ubah setiap unit retrievable menjadi vektor menggunakan Gemini Embeddings API.

In [7]:
def get_embedding(text):
    result = client.models.embed_content(
        model='gemini-embedding-001',  # sesuaikan versi model embedding
        contents=text
    )
    return result.embeddings[0].values

embeddings = [get_embedding(doc) for doc in documents]
embeddings = np.array(embeddings).astype('float32')
print('Shape embeddings:', embeddings.shape)

Shape embeddings: (59, 3072)


## 4. Menyimpan Embeddings ke Vector Database (FAISS)

In [8]:
dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(embeddings)

print('Jumlah vektor tersimpan di index:', index.ntotal)

Jumlah vektor tersimpan di index: 59


In [9]:
assert index.ntotal == len(df), 'Jumlah vektor tidak sama dengan jumlah entri!'
print('OK: ', index.ntotal, 'vektor =', len(df), 'entri FAQ')

os.makedirs('../vector_db', exist_ok=True)
faiss.write_index(index, '../vector_db/faq_index.faiss')
np.save('../vector_db/embeddings.npy', embeddings)
print('Index dan embeddings tersimpan di folder vector_db/')

OK:  59 vektor = 59 entri FAQ
Index dan embeddings tersimpan di folder vector_db/


## 5. Mengimplementasikan Semantic Search

Fungsi untuk mengambil beberapa entri FAQ paling relevan berdasarkan query pengguna.

In [10]:
def semantic_search(query, top_k=3):
    query_embedding = np.array([get_embedding(query)]).astype('float32')
    distances, indices = index.search(query_embedding, top_k)

    results = []
    for rank, idx in enumerate(indices[0]):
        results.append({
            'rank': rank + 1,
            'question': df.iloc[idx]['question'],
            'answer': df.iloc[idx]['answer'],
            'category': df.iloc[idx]['category'],
            'distance': distances[0][rank]
        })
    return results

# Contoh percobaan
test_results = semantic_search('kenapa paket saya belum sampai?')
for r in test_results:
    print(r)

{'rank': 1, 'question': 'Bagaimana jika paket saya belum sampai melebihi estimasi?', 'answer': 'Jika paket melebihi estimasi pengiriman lebih dari 3 hari, silakan hubungi customer service dengan menyertakan nomor resi untuk ditelusuri lebih lanjut.', 'category': 'Pengiriman', 'distance': np.float32(0.33353508)}
{'rank': 2, 'question': 'Kenapa status pengiriman saya tidak berubah selama beberapa hari?', 'answer': 'Status yang tidak berubah bisa disebabkan keterlambatan update sistem kurir. Disarankan menunggu 1x24 jam sebelum melaporkan ke customer service.', 'category': 'Pengiriman', 'distance': np.float32(0.4088098)}
{'rank': 3, 'question': 'Kenapa nomor resi saya belum bisa dilacak di website kurir?', 'answer': "Nomor resi baru dapat dilacak setelah paket diserahterimakan ke pihak kurir, biasanya butuh waktu beberapa jam setelah status 'Dikirim'.", 'category': 'Pengiriman', 'distance': np.float32(0.4320181)}


In [11]:
def tampilkan_hasil(query, top_k=3):
    print('Query:', query)
    for r in semantic_search(query, top_k):
        print(f"  #{r['rank']} (jarak {r['distance']:.4f}) [{r['category']}] {r['question']}")
    print()

tampilkan_hasil('kenapa paket saya belum sampai?')
tampilkan_hasil('barang yang saya terima rusak, bisa dikembalikan?')
tampilkan_hasil('apakah ada promo hari ini?')

Query: kenapa paket saya belum sampai?
  #1 (jarak 0.3335) [Pengiriman] Bagaimana jika paket saya belum sampai melebihi estimasi?
  #2 (jarak 0.4088) [Pengiriman] Kenapa status pengiriman saya tidak berubah selama beberapa hari?
  #3 (jarak 0.4320) [Pengiriman] Kenapa nomor resi saya belum bisa dilacak di website kurir?

Query: barang yang saya terima rusak, bisa dikembalikan?
  #1 (jarak 0.3510) [Pengembalian] Bagaimana prosedur pengembalian barang yang tidak sesuai?
  #2 (jarak 0.4057) [Pengembalian] Apakah semua produk bisa dikembalikan?
  #3 (jarak 0.4117) [Pengiriman] Apa yang harus dilakukan jika paket diterima dalam kondisi rusak?

Query: apakah ada promo hari ini?
  #1 (jarak 0.3920) [Promo] Bagaimana cara mendapatkan info promo terbaru?
  #2 (jarak 0.4762) [Promo] Apakah promo bisa digabung dengan diskon member?
  #3 (jarak 0.5347) [Promo] Apakah member baru mendapat diskon khusus?



## 6. Mendesain Prompt dan Context

Susun prompt terstruktur (Persona, Task, Context, Format) yang menyisipkan hasil semantic search sebagai bagian Context, sebelum dikirim ke LLM.

In [12]:
def build_prompt(query, search_results):
    context = '\n\n'.join(
        f"FAQ: {r['question']}\nJawaban: {r['answer']}" for r in search_results
    )
    prompt = f'''Kamu adalah asisten customer service e-commerce yang ramah dan membantu.

Tugas: Jawab pertanyaan pengguna HANYA berdasarkan informasi FAQ berikut. Jika informasi tidak ditemukan dalam FAQ, katakan bahwa kamu belum memiliki informasi tersebut dan sarankan menghubungi customer service.

Konteks FAQ relevan:
{context}

Pertanyaan pengguna: {query}

Jawaban:'''
    return prompt

## 7. Mengintegrasikan LLM

In [13]:
def generate_answer(query, top_k=3):
    search_results = semantic_search(query, top_k=top_k)
    prompt = build_prompt(query, search_results)

    response = client.models.generate_content(
        model='gemini-3-flash-preview',  # sesuaikan versi model
        contents=prompt,
        config=types.GenerateContentConfig(temperature=0.3)
    )
    return response.text, search_results

answer, sources = generate_answer('apakah bisa bayar pakai cicilan?')
print('Jawaban LLM:')
print(answer)
print()
print('Sumber FAQ yang digunakan:')
for s in sources:
    print('-', s['question'])

Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


Jawaban LLM:
Halo! Tentu, fitur cicilan tersedia untuk transaksi dengan kartu kredit tertentu pada nominal minimal yang ditentukan oleh pihak bank. Ada lagi yang bisa saya bantu?

Sumber FAQ yang digunakan:
- Apakah saya bisa membayar dengan cicilan?
- Apakah bisa membayar menggunakan lebih dari satu metode sekaligus?
- Metode pembayaran apa saja yang tersedia?


## 8. Menguji Relevansi Hasil Retrieval dan Response AI

Uji sistem dengan beberapa query berbeda, termasuk query yang menggunakan kata/frasa berbeda dari FAQ asli (sinonim/parafrase) untuk menilai apakah semantic search benar-benar menangkap makna, bukan hanya mencocokkan kata kunci.

_Uji minimal 5 query berbeda, termasuk:_
- _1-2 query yang jelas ada jawabannya di FAQ_
- _1-2 query dengan kata-kata berbeda dari FAQ asli (parafrase/sinonim)_
- _1 query yang TIDAK ada jawabannya di FAQ (uji bagaimana LLM merespons ketidaktahuannya)_

In [15]:
test_queries = [
    ("Ada di FAQ", "Apakah saya bisa membayar dengan cicilan?"),
    ("Ada di FAQ", "Bagaimana jika paket saya belum sampai melebihi estimasi?"),
    ("Parafrase", "Boleh nggak bayarnya dicicil per bulan pakai kartu kredit?"),
    ("Parafrase", "Barang yang datang pecah, uangnya bisa dikembalikan nggak?"),
    ("Tidak ada di FAQ", "Bagaimana cara mendaftar jadi penjual di platform ini?"),
    ("topik mirip", "Apakah ada promo hari ini?"),
]

for tipe, q in test_queries:
    answer, sources = generate_answer(q)
    print(f'[{tipe}] Query: {q}')
    print('Jawaban:', answer)
    print(f"Sumber teratas: {sources[0]['question']} (jarak {sources[0]['distance']:.4f})")
    print('---')

[Ada di FAQ] Query: Apakah saya bisa membayar dengan cicilan?
Jawaban: Halo! Tentu, fitur cicilan tersedia untuk transaksi dengan kartu kredit tertentu pada nominal minimal yang ditentukan oleh pihak bank. Ada lagi yang bisa saya bantu?
Sumber teratas: Apakah saya bisa membayar dengan cicilan? (jarak 0.2049)
---
[Ada di FAQ] Query: Bagaimana jika paket saya belum sampai melebihi estimasi?
Jawaban: Halo! Jika paket Anda belum sampai dan sudah melebihi estimasi pengiriman lebih dari 3 hari, silakan hubungi customer service kami dengan menyertakan nomor resi untuk ditelusuri lebih lanjut. Ada lagi yang bisa saya bantu?
Sumber teratas: Bagaimana jika paket saya belum sampai melebihi estimasi? (jarak 0.1612)
---
[Parafrase] Query: Boleh nggak bayarnya dicicil per bulan pakai kartu kredit?
Jawaban: Halo! Tentu bisa. Fitur cicilan tersedia untuk transaksi dengan kartu kredit tertentu pada nominal minimal yang ditentukan oleh pihak bank.
Sumber teratas: Apakah saya bisa membayar dengan cicilan

### Catatan Analisis Relevansi

_Tuliskan pengamatanmu di sini: query mana yang berhasil di-retrieve dengan baik, query mana yang kurang relevan hasilnya, dan bagaimana LLM merespons ketika informasi tidak tersedia di knowledge base._

### Catatan Analisis Relevansi

dari hasil uji 6 query. Jarak yang dipakai adalah L2, jadi makin kecil makin mirip.

**Retrieval yang berhasil**
- Query yang ada di FAQ menemukan FAQ yang sama persis, dengan jarak paling kecil (0.20 dan 0.16).
- Query parafrase tetap benar walau beda kata. "Dicicil per bulan" menemukan FAQ cicilan (0.25), dan "pecah" menemukan FAQ paket rusak (0.45).
- Jadi semantic search menangkap makna, bukan cuma kata kunci.

**Retrieval yang kurang pas**
- Query "daftar jadi penjual" menemukan FAQ "daftar akun baru" (0.51). Kelihatan mirip, tapi maksudnya beda, dan memang tidak ada FAQ soal penjual.

**Respons LLM**
- Kalau info ada di FAQ, jawabannya sesuai dan tidak mengarang.
- Kalau info tidak ada (query penjual), LLM jujur bilang belum punya info dan mengarahkan ke customer service.
- Query "pecah": LLM menjawab cara melapor barang rusak, lalu bilang info pengembalian uang tidak ada di FAQ.
- Query "promo hari ini": LLM bilang tidak punya info daftar promo harian di FAQ, lalu mengarahkan ke customer service.

**Keterbatasan**
- Jarak tidak bisa jadi patokan relevan atau tidak. Query yang tidak ada di FAQ (0.51) hampir sama dengan parafrase yang benar (0.45).
- Top 3 selalu muncul walau tidak ada yang cocok, jadi LLM yang menentukan info itu ada atau tidak.
- Jawaban LLM kadang lebih panjang dari batas 3-4 kalimat di prompt.
- Jawaban LLM bisa sedikit beda tiap dijalankan (isinya sama, kata-katanya berubah), sedangkan hasil retrieval dan jaraknya tetap sama.

**Kesimpulan:** pipeline berjalan dari awal sampai akhir, dan semantic search cukup bisa diandalkan untuk query yang ada di FAQ, termasuk yang kata-katanya berbeda. Kalau pertanyaannya tidak ada di FAQ, hasil pencarian bisa menyesatkan, tapi LLM masih bisa menjawab dengan jujur karena prompt-nya membatasi jawaban hanya dari FAQ.